In [19]:
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Pemilihan model
from sklearn.model_selection import train_test_split, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Menangani data yang tidak seimbang dengan undersampling
from imblearn.under_sampling import RandomUnderSampler

In [20]:
df= pd.read_csv('Clean Dataset/fraudTrain_dataset_cleaned.csv')

df.head()

,merchant,category,amt,gender,street,city,state,zip,city_pop,job,is_fraud,age,day_of_week,transaction_min,transaction_hour,transaction_date,transaction_month,full_name,transaction_distance
0,"Rippin, Kub and Mann",Misc Net,4.97,Female,561 Perry Cove,Moravian Falls,North Carolina,28654,3495,"Psychologist, counselling",0,31,Tuesday,0,0,1,1,Jennifer Banks,78.597568
1,"Heller, Gutmann and Zieme",Grocery Pos,107.23,Female,43039 Riley Greens Suite 393,Orient,Washington,99160,149,Special educational needs teacher,0,41,Tuesday,0,0,1,1,Stephanie Gill,30.212176
2,Lind-Buckridge,Entertainment,220.11,Male,594 White Dale Suite 530,Malad City,Idaho,83252,4154,Nature conservation officer,0,57,Tuesday,0,0,1,1,Edward Sanchez,108.206083
3,"Kutch, Hermiston and Farrell",Gas Transport,45.00,Male,9443 Cynthia Court Apt. 038,Boulder,Montana,59632,1939,Patent attorney,0,52,Tuesday,1,0,1,1,Jeremy White,95.673231
4,Keeling-Crist,Misc Pos,41.96,Male,408 Bradley Rest,Doe Hill,Virginia,24433,99,Dance movement psychotherapist,0,33,Tuesday,3,0,1,1,Tyler Garcia,77.556744


In [21]:
# Membagi data menjadi X dan y
X = df.drop("is_fraud", axis = 1)
y = df["is_fraud"]

In [22]:
# Memeriksa tipe data object
df.select_dtypes(include='object').columns

Index(['merchant', 'category', 'gender', 'street', 'city', 'state', 'job',
       'day_of_week', 'full_name'],
      dtype='object')

In [23]:
# Memfilter kolom kategoris
categorical_col_names = ["merchant", "category", "gender","street", "city", "state", "job", "day_of_week", "full_name"]

In [24]:
# Membagi data menjadi train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 101)
(X_train.shape, y_train.shape), (X_test.shape, y_test.shape)

(((1037340, 18), (1037340,)), ((259335, 18), (259335,)))

In [25]:
# Memeriksa jumlah instance untuk setiap kelas dalam set pelatihan
print(f"Jumlah untuk Kelas Negatif: {len(y_train) - sum(y_train)}")
print(f"Jumlah untuk Kelas Positif: {sum(y_train)}")

Jumlah untuk Kelas Negatif: 1031335
Jumlah untuk Kelas Positif: 6005


In [26]:
# Menggunakan OrdinalEncoder untuk mengkodekan data kategorikal
encoder = OrdinalEncoder()
X_train.loc[:, categorical_col_names] = encoder.fit_transform(X_train.loc[:, categorical_col_names])
X_test.loc[:, categorical_col_names] = encoder.transform(X_test.loc[:, categorical_col_names])

X_train.head()

,merchant,category,amt,gender,street,city,state,zip,city_pop,job,age,day_of_week,transaction_min,transaction_hour,transaction_date,transaction_month,full_name,transaction_distance
399250,649.0,11.0,4.49,0.0,934.0,806.0,16.0,66618,163415,416.0,15,3.0,5,20,30,6,265.0,79.609006
1135802,207.0,5.0,47.76,0.0,416.0,391.0,0.0,36749,1089,286.0,49,2.0,9,15,18,4,78.0,107.895967
981086,449.0,10.0,44.02,0.0,170.0,546.0,36.0,73559,540,392.0,37,3.0,32,20,2,2,775.0,115.130327
756474,175.0,11.0,1.52,0.0,160.0,349.0,35.0,44233,7646,291.0,31,5.0,54,12,19,11,897.0,33.123855
37034,111.0,6.0,91.25,1.0,698.0,718.0,32.0,14778,1453,475.0,45,5.0,55,12,22,1,23.0,87.827720


In [27]:
# Menggunakan StandardScaler untuk menskalakan data numerik
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [28]:
# Memakai RandomUnderSampler untuk undersampling data
under_sampler = RandomUnderSampler(random_state=101)
X_train_resampled, y_train_resampled = under_sampler.fit_resample(X_train, y_train)
(X_train_resampled.shape, y_train_resampled.shape)

((12010, 18), (12010,))

In [29]:
# Memeriksa jumlah instance untuk setiap kelas dalam set pelatihan setelah resampling
print(f"Jumlah untuk Kelas Negatif: {len(y_train_resampled) - sum(y_train_resampled)}")
print(f"Jumlah untuk Kelas Positif: {sum(y_train_resampled)}")

Jumlah untuk Kelas Negatif: 6005
Jumlah untuk Kelas Positif: 6005


In [30]:
# Kamus model yang akan diuji
models = {
    "LogisticRegression" : LogisticRegression(C=0.01, solver = "saga", class_weight="balanced", random_state=101),
    "RandomForestClassifier" : RandomForestClassifier(n_estimators=50, max_depth=7, class_weight="balanced", random_state=101),
    "XGBClassifier" : XGBClassifier(n_estimators=50, max_depth=7, enable_categorical = True, random_state = 101)
}

In [31]:
# Inisialisasi kamus untuk menyimpan kinerja model
model_performance = {}

# Melakukan pengulangan melalui setiap model, pelatihan, dan mengevaluasi kinerja
for model_name, model in models.items():
    print(f"Training {model_name}")

    # Melatih model
    model.fit(X_train_resampled, y_train_resampled)
    
    # Membuat prediksi pada set pengujian
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]  # Untuk ROC-AUC
    
    # Pelatihan dan Pengujian Skor CV
    training_cv_score = cross_val_score(model, X_train_resampled, y_train_resampled, scoring = "roc_auc")
    testing_cv_score = cross_val_score(model, X_test, y_test, scoring = "roc_auc")
    
    # Mengevaluasi model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    TN, FP, FN, TP = confusion_matrix(y_test, y_pred).ravel()
    
    # Menyimpan metrik kinerja untuk setiap model
    model_performance[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC AUC': roc_auc,
        "TP" : TP,
        "TN" : TN,
        "FP" : FP,
        "FN" : FN,
        "Avg Training CV Score" : np.mean(training_cv_score),
        "Avg Testing CV Score" : np.mean(testing_cv_score),
        "Trainning CV Score Std" : np.std(training_cv_score),
        "Testing CV Score Std" : np.std(testing_cv_score)
    }
    
    print(f"Done Training {model_name}\n")

Training LogisticRegression


LogisticRegression(C=0.01, class_weight='balanced', random_state=101,
                   solver='saga')

Done Training LogisticRegression

Training RandomForestClassifier


RandomForestClassifier(class_weight='balanced', max_depth=7, n_estimators=50,
                       random_state=101)

Done Training RandomForestClassifier

Training XGBClassifier


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

Done Training XGBClassifier



In [32]:
# Memeriksa kinerja berbagai model
model_performance_df = pd.DataFrame(model_performance)
model_performance_df

,LogisticRegression,RandomForestClassifier,XGBClassifier
Accuracy,0.949583,0.964509,0.975946
Precision,0.082045,0.133644,0.191802
Recall,0.756829,0.936043,0.982012
F1-Score,0.148042,0.233894,0.320923
ROC AUC,0.860063,0.985504,0.997448
TP,1136.000000,1405.000000,1474.000000
TN,245124.000000,248726.000000,251623.000000
FP,12710.000000,9108.000000,6211.000000
FN,365.000000,96.000000,27.000000
Avg Training CV Score,0.859139,0.984031,0.997212


In [33]:
import pickle

# Ambil model XGBoost dari dictionary models
xgboost = models['XGBClassifier']

# Simpan model ke dalam file .pkl
with open('xgboost_fraud_model.pkl', 'wb') as file:
    pickle.dump(xgboost, file)  

print("✅ xgboost_fraud_model.pkl berhasil di simpan")

# Simpan encoder setelah fit
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("✅ encoder.pkl berhasil di simpan")

# Simpan scaler setelah fit
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ scaler.pkl berhasil di simpan")

✅ xgboost_fraud_model.pkl berhasil di simpan
✅ encoder.pkl berhasil di simpan
✅ scaler.pkl berhasil di simpan


In [34]:
# 1. Membaca file CSV testing
df_test = pd.read_csv("Clean Dataset/fraudTest_dataset_cleaned.csv")  # Ganti dengan nama file kamu

In [35]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

print("=== PREDIKSI MODEL PADA DATA TESTING ===\n")

for model_name, model in models.items():
    print(f"🔍 Evaluasi Model: {model_name}")
    
    # Prediksi data testing
    y_pred = model.predict(X_test)
    
    # Hitung jumlah fraud (positif = 1) yang berhasil ditangkap dan yang miss
    TN, FP, FN, TP = confusion_matrix(y_test, y_pred).ravel()
    
    total_test = len(y_test)
    total_fraud = sum(y_test)
    
    print(f"Total transaksi di data testing       : {total_test}")
    print(f"Total transaksi fraud (aktual)        : {total_fraud}")
    print(f"✅ Fraud berhasil ditangkap (TP)      : {TP}")
    print(f"❌ Fraud yang terlewatkan (FN)        : {FN}")
    print(f"🎯 Akurasi deteksi fraud              : {TP}/{total_fraud} ({(TP/total_fraud)*100:.2f}%)")
    print("-" * 50)

=== PREDIKSI MODEL PADA DATA TESTING ===

🔍 Evaluasi Model: LogisticRegression
Total transaksi di data testing       : 259335
Total transaksi fraud (aktual)        : 1501
✅ Fraud berhasil ditangkap (TP)      : 1136
❌ Fraud yang terlewatkan (FN)        : 365
🎯 Akurasi deteksi fraud              : 1136/1501 (75.68%)
--------------------------------------------------
🔍 Evaluasi Model: RandomForestClassifier
Total transaksi di data testing       : 259335
Total transaksi fraud (aktual)        : 1501
✅ Fraud berhasil ditangkap (TP)      : 1405
❌ Fraud yang terlewatkan (FN)        : 96
🎯 Akurasi deteksi fraud              : 1405/1501 (93.60%)
--------------------------------------------------
🔍 Evaluasi Model: XGBClassifier
Total transaksi di data testing       : 259335
Total transaksi fraud (aktual)        : 1501
✅ Fraud berhasil ditangkap (TP)      : 1474
❌ Fraud yang terlewatkan (FN)        : 27
🎯 Akurasi deteksi fraud              : 1474/1501 (98.20%)
-------------------------------------